# HSCA Recipe Extraction Analysis
## Comprehensive Analysis of Improved Extraction Results

This notebook analyzes the results from our improved recipe extraction system, including:
- Enhanced OCR correction
- Cross-referencing with existing recipes
- Duplicate detection and filtering
- Quality metrics and insights

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
from pathlib import Path

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("📊 Recipe Extraction Analysis Notebook Loaded")
print("=" * 50)

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'seaborn'

## 1. Load and Examine Extraction Results

In [ ]:
# Load extraction results
def load_data():
    data = {}
    
    # Load enhanced extraction results
    try:
        with open('enhanced_extracted_recipes/enhanced_hsca_recipes.json', 'r') as f:
            data['enhanced'] = json.load(f)
        print("✅ Enhanced extraction results loaded")
    except FileNotFoundError:
        print("⚠️  Enhanced extraction results not found")
    
    # Load cross-reference results
    try:
        with open('cross_reference_report.json', 'r') as f:
            data['cross_ref'] = json.load(f)
        print("✅ Cross-reference results loaded")
    except FileNotFoundError:
        print("⚠️  Cross-reference results not found")
    
    # Load filtered results
    try:
        with open('enhanced_extracted_recipes/filtered_hsca_recipes.json', 'r') as f:
            data['filtered'] = json.load(f)
        print("✅ Filtered results loaded")
    except FileNotFoundError:
        print("⚠️  Filtered results not found")
    
    # Load pipeline report
    try:
        with open('enhanced_extracted_recipes/pipeline_report.json', 'r') as f:
            data['pipeline'] = json.load(f)
        print("✅ Pipeline report loaded")
    except FileNotFoundError:
        print("⚠️  Pipeline report not found")
    
    # Load existing recipes database
    try:
        with open('existing_recipes_db.json', 'r') as f:
            data['existing'] = json.load(f)
        print("✅ Existing recipes database loaded")
    except FileNotFoundError:
        print("⚠️  Existing recipes database not found")
    
    return data

# Load all data
data = load_data()

print(f"\n📈 Data Summary:")
for key, value in data.items():
    if isinstance(value, dict):
        print(f"  {key}: {len(value)} top-level keys")
    else:
        print(f"  {key}: {type(value)}")

## 2. Pipeline Performance Overview

In [ ]:
# Display pipeline performance metrics
if 'pipeline' in data:
    pipeline = data['pipeline']
    
    print("🎯 PIPELINE PERFORMANCE SUMMARY")
    print("=" * 40)
    
    # Extraction metrics
    if 'extraction_results' in pipeline:
        results = pipeline['extraction_results']
        
        print(f"📊 Extraction Results:")
        if 'extracted_count' in results:
            print(f"  • Total recipes extracted: {results['extracted_count']}")
        
        if 'cross_reference' in results:
            cr = results['cross_reference']
            print(f"  • Matches found: {cr.get('total_matches', 0)}")
            print(f"  • Duplicates identified: {cr.get('duplicates', 0)}")
            print(f"  • High confidence matches: {cr.get('high_confidence', 0)}")
            print(f"  • New recipes: {cr.get('no_matches', 0)}")
            print(f"  • Duplicate rate: {cr.get('duplicate_percentage', 0):.1f}%")
        
        if 'filtering' in results:
            filt = results['filtering']
            print(f"  • Recipes after filtering: {filt.get('filtered_recipes', 0)}")
            print(f"  • Duplicates removed: {filt.get('duplicates_removed', 0)}")
    
    # Quality metrics
    if 'quality_metrics' in pipeline:
        metrics = pipeline['quality_metrics']
        print(f"\n📈 Quality Metrics:")
        for metric, value in metrics.items():
            if isinstance(value, float):
                print(f"  • {metric.replace('_', ' ').title()}: {value:.1f}%")
    
    # Recommendations
    if 'recommendations' in pipeline:
        recs = pipeline['recommendations']
        print(f"\n💡 Recommendations:")
        for rec in recs:
            print(f"  • {rec['message']} ({rec['priority']})")

else:
    print("⚠️  Pipeline data not available for analysis")

## 3. Extracted Recipes Analysis

In [ ]:
# Analyze extracted recipes
if 'enhanced' in data and 'extracted_recipes' in data['enhanced']:
    recipes = data['enhanced']['extracted_recipes']
    
    print(f"🍳 EXTRACTED RECIPES ANALYSIS")
    print("=" * 40)
    print(f"Total recipes extracted: {len(recipes)}")
    
    if len(recipes) > 0:
        # Analyze recipe structure
        ingredient_counts = []
        instruction_counts = []
        categories = []
        lessons = []
        
        for recipe in recipes:
            recipe_data = recipe.get('recipe', {})
            
            # Count ingredients
            ingredients = recipe_data.get('ingredients', [])
            ingredient_counts.append(len(ingredients))
            
            # Count instructions
            instructions = recipe_data.get('instructions', [])
            instruction_counts.append(len(instructions))
            
            # Collect categories
            category = recipe.get('category', 'unknown')
            categories.append(category)
            
            # Collect lessons
            lesson = recipe.get('lesson', 'unknown')
            lessons.append(lesson)
        
        # Display statistics
        print(f"\n📊 Recipe Statistics:")
        print(f"  • Average ingredients per recipe: {sum(ingredient_counts)/len(ingredient_counts):.1f}")
        print(f"  • Average instructions per recipe: {sum(instruction_counts)/len(instruction_counts):.1f}")
        print(f"  • Recipes with >5 ingredients: {sum(1 for c in ingredient_counts if c > 5)}")
        print(f"  • Recipes with >5 instructions: {sum(1 for c in instruction_counts if c > 5)}")
        
        # Category distribution
        category_counts = Counter(categories)
        print(f"\n🏷️  Category Distribution:")
        for category, count in category_counts.most_common():
            print(f"  • {category}: {count} recipes")
        
        # Lesson distribution (top 10)
        lesson_counts = Counter(lessons)
        print(f"\n📚 Top Lessons by Recipe Count:")
        for lesson, count in lesson_counts.most_common(10):
            print(f"  • {lesson}: {count} recipes")
    
else:
    print("⚠️  No extracted recipes found for analysis")

## 4. Cross-Reference Analysis

In [ ]:
# Analyze cross-reference results
if 'cross_ref' in data:
    cross_ref = data['cross_ref']
    
    print(f"🔗 CROSS-REFERENCE ANALYSIS")
    print("=" * 40)
    
    # Overall statistics
    stats = cross_ref.get('statistics', {})
    print(f"Total extracted recipes analyzed: {cross_ref.get('total_extracted', 0)}")
    print(f"Total existing recipes in database: {cross_ref.get('total_existing', 0)}")
    print(f"Matches found: {stats.get('total_matches', 0)}")
    print(f"Duplicates identified: {stats.get('duplicates', 0)}")
    print(f"High confidence matches: {stats.get('high_confidence', 0)}")
    print(f"No matches (new recipes): {stats.get('no_matches', 0)}")
    print(f"Duplicate percentage: {stats.get('duplicate_percentage', 0):.1f}%")
    
    # Show example duplicates
    if 'duplicates' in cross_ref and len(cross_ref['duplicates']) > 0:
        print(f"\n🔍 Example Duplicates Found:")
        for i, duplicate in enumerate(cross_ref['duplicates'][:5]):
            extracted = duplicate['extracted_name']
            existing = duplicate['existing_name']
            confidence = duplicate['confidence_score']
            print(f"  {i+1}. '{extracted}' → '{existing}' ({confidence:.3f})")
    
    # Show example high confidence matches
    if 'high_confidence_matches' in cross_ref and len(cross_ref['high_confidence_matches']) > 0:
        print(f"\n🎯 Example High Confidence Matches:")
        for i, match in enumerate(cross_ref['high_confidence_matches'][:5]):
            extracted = match['extracted_name']
            existing = match['existing_name']
            confidence = match['confidence_score']
            print(f"  {i+1}. '{extracted}' → '{existing}' ({confidence:.3f})")
    
    # Show example new recipes
    if 'no_matches' in cross_ref and len(cross_ref['no_matches']) > 0:
        print(f"\n🆕 Example New Recipes (No Matches):")
        for i, no_match in enumerate(cross_ref['no_matches'][:10]):
            recipe_name = no_match['recipe_name']
            category = no_match.get('suggested_category', 'unknown')
            print(f"  {i+1}. '{recipe_name}' ({category})")

else:
    print("⚠️  Cross-reference data not available for analysis")

## 5. OCR Correction Examples

In [ ]:
# Analyze OCR corrections and show examples
if 'cross_ref' in data and 'duplicates' in data['cross_ref']:
    print(f"🔧 OCR CORRECTION EXAMPLES")
    print("=" * 40)
    
    duplicates = data['cross_ref']['duplicates']
    
    print(f"The following examples show how our OCR correction system")
    print(f"successfully identified recipes despite character corruption:\n")
    
    ocr_examples = []
    
    for duplicate in duplicates:
        extracted = duplicate['extracted_name']
        existing = duplicate['existing_name']
        confidence = duplicate['confidence_score']
        
        # Identify OCR corruption patterns
        ocr_issues = []
        
        # Check for common OCR patterns
        if '0' in extracted and 'O' in existing:
            ocr_issues.append("0→O substitution")
        if '1' in extracted and 'I' in existing:
            ocr_issues.append("1→I substitution")
        if '5' in extracted and 'S' in existing:
            ocr_issues.append("5→S substitution")
        
        # Character count differences
        char_diff = abs(len(extracted) - len(existing))
        if char_diff > 2:
            ocr_issues.append(f"Length difference: {char_diff} chars")
        
        ocr_examples.append({
            'extracted': extracted,
            'existing': existing,
            'confidence': confidence,
            'issues': ocr_issues
        })
    
    for i, example in enumerate(ocr_examples[:10]):
        print(f"{i+1}. OCR Corrupted: '{example['extracted']}'")
        print(f"   Clean Version:  '{example['existing']}'")
        print(f"   Confidence:     {example['confidence']:.3f}")
        if example['issues']:
            print(f"   OCR Issues:     {', '.join(example['issues'])}")
        print()

else:
    print("⚠️  OCR correction examples not available")

## 6. Sample Recipe Display

In [ ]:
# Display sample recipes to show extraction quality
if 'enhanced' in data and 'extracted_recipes' in data['enhanced']:
    recipes = data['enhanced']['extracted_recipes']
    
    print(f"🍳 SAMPLE EXTRACTED RECIPES")
    print("=" * 50)
    
    # Show first few recipes with good structure
    good_recipes = []
    for recipe in recipes:
        recipe_data = recipe.get('recipe', {})
        ingredients = recipe_data.get('ingredients', [])
        instructions = recipe_data.get('instructions', [])
        
        # Only show recipes with reasonable content
        if len(ingredients) >= 3 and len(instructions) >= 3:
            good_recipes.append(recipe)
    
    # Display up to 3 sample recipes
    for i, recipe in enumerate(good_recipes[:3]):
        recipe_data = recipe.get('recipe', {})
        
        print(f"\n📖 Recipe {i+1}: {recipe_data.get('name', 'Unknown')}")
        print(f"Category: {recipe.get('category', 'unknown')}")
        print(f"Lesson: {recipe.get('lesson', 'unknown')}")
        
        # Display description if available
        if 'description' in recipe_data:
            description = recipe_data['description'][:200]
            print(f"Description: {description}{'...' if len(recipe_data['description']) > 200 else ''}")
        
        # Display yield if available
        if 'yield' in recipe_data:
            print(f"Yield: {recipe_data['yield']}")
        
        # Show ingredients
        ingredients = recipe_data.get('ingredients', [])
        print(f"\nIngredients ({len(ingredients)}):")
        for j, ingredient in enumerate(ingredients[:10]):  # Show first 10
            amount = ingredient.get('amount', '')
            unit = ingredient.get('unit', '')
            name = ingredient.get('name', '')
            print(f"  {j+1}. {amount} {unit} {name}".strip())
        
        if len(ingredients) > 10:
            print(f"  ... and {len(ingredients) - 10} more")
        
        # Show instructions
        instructions = recipe_data.get('instructions', [])
        print(f"\nInstructions ({len(instructions)}):")
        for j, instruction in enumerate(instructions[:5]):  # Show first 5
            print(f"  {j+1}. {instruction[:100]}{'...' if len(instruction) > 100 else ''}")
        
        if len(instructions) > 5:
            print(f"  ... and {len(instructions) - 5} more steps")
        
        print("-" * 50)
    
    if len(good_recipes) == 0:
        print("⚠️  No recipes with sufficient ingredients and instructions found")
    
else:
    print("⚠️  No extracted recipes available for display")

## 7. Comparison with Existing Database

In [ ]:
# Compare extracted recipes with existing database
if 'existing' in data:
    existing = data['existing']
    
    print(f"📚 COMPARISON WITH EXISTING DATABASE")
    print("=" * 50)
    
    # Analyze existing database
    existing_categories = {}
    for recipe_key, recipe_info in existing.items():
        category = recipe_info.get('category', 'unknown')
        if category not in existing_categories:
            existing_categories[category] = 0
        existing_categories[category] += 1
    
    print(f"Existing Database:")
    print(f"  • Total recipes: {len(existing)}")
    print(f"  • Categories: {len(existing_categories)}")
    
    print(f"\nExisting Categories:")
    for category, count in sorted(existing_categories.items()):
        print(f"  • {category}: {count} recipes")
    
    # Compare with extracted if available
    if 'enhanced' in data and 'extracted_recipes' in data['enhanced']:
        extracted_count = len(data['enhanced']['extracted_recipes'])
        existing_count = len(existing)
        
        print(f"\n📊 Database Comparison:")
        print(f"  • Existing recipes: {existing_count}")
        print(f"  • Newly extracted: {extracted_count}")
        print(f"  • Total potential: {existing_count + extracted_count}")
        
        if 'cross_ref' in data and 'statistics' in data['cross_ref']:
            stats = data['cross_ref']['statistics']
            new_recipes = stats.get('no_matches', 0)
            duplicates = stats.get('duplicates', 0)
            
            print(f"  • True new recipes: {new_recipes}")
            print(f"  • Confirmed duplicates: {duplicates}")
            print(f"  • Net database growth: {new_recipes} recipes")
            
            if existing_count > 0:
                growth_rate = (new_recipes / existing_count) * 100
                print(f"  • Growth rate: {growth_rate:.1f}%")

else:
    print("⚠️  Existing database not available for comparison")

## 8. Quality Assessment & Recommendations

In [ ]:
# Comprehensive quality assessment
print(f"✅ QUALITY ASSESSMENT & RECOMMENDATIONS")
print("=" * 50)

# Overall assessment
assessment = {
    'extraction_working': False,
    'cross_ref_working': False,
    'filtering_working': False,
    'quality_score': 0
}

# Check if extraction is working
if 'enhanced' in data and 'extracted_recipes' in data['enhanced']:
    recipe_count = len(data['enhanced']['extracted_recipes'])
    if recipe_count > 0:
        assessment['extraction_working'] = True
        assessment['quality_score'] += 30
        print(f"✅ Extraction: Working ({recipe_count} recipes found)")
    else:
        print(f"❌ Extraction: Not working (0 recipes found)")
else:
    print(f"❌ Extraction: No data available")

# Check if cross-referencing is working
if 'cross_ref' in data and 'statistics' in data['cross_ref']:
    stats = data['cross_ref']['statistics']
    if stats.get('total_matches', 0) > 0:
        assessment['cross_ref_working'] = True
        assessment['quality_score'] += 30
        duplicate_rate = stats.get('duplicate_percentage', 0)
        print(f"✅ Cross-referencing: Working ({duplicate_rate:.1f}% duplicate rate)")
    else:
        print(f"❌ Cross-referencing: Not finding matches")
else:
    print(f"❌ Cross-referencing: No data available")

# Check if filtering is working
if 'filtered' in data and 'summary' in data['filtered']:
    summary = data['filtered']['summary']
    filtered_count = summary.get('filtered_recipes', 0)
    removed_count = summary.get('duplicates_removed', 0)
    if filtered_count > 0:
        assessment['filtering_working'] = True
        assessment['quality_score'] += 20
        print(f"✅ Filtering: Working ({filtered_count} kept, {removed_count} removed)")
    else:
        print(f"❌ Filtering: No recipes retained")
else:
    print(f"❌ Filtering: No data available")

# Pipeline integration check
if 'pipeline' in data:
    assessment['quality_score'] += 20
    print(f"✅ Pipeline: Integrated successfully")
else:
    print(f"❌ Pipeline: Not integrated")

print(f"\n🎯 Overall Quality Score: {assessment['quality_score']}/100")

# Generate recommendations
print(f"\n💡 Recommendations:")

if assessment['quality_score'] >= 80:
    print(f"  ✅ System is working well! Ready for production use.")
    print(f"  🚀 Consider running full extraction on all 507 pages.")
elif assessment['quality_score'] >= 60:
    print(f"  ⚠️  System is mostly working but needs refinement.")
    print(f"  🔧 Focus on improving extraction accuracy.")
elif assessment['quality_score'] >= 40:
    print(f"  ❌ System has significant issues that need addressing.")
    print(f"  🔍 Debug extraction and cross-referencing components.")
else:
    print(f"  🚨 System is not functioning properly.")
    print(f"  🛠️  Requires comprehensive debugging and fixes.")

# Specific recommendations based on data
if 'pipeline' in data and 'recommendations' in data['pipeline']:
    pipeline_recs = data['pipeline']['recommendations']
    if pipeline_recs:
        print(f"\n🎯 Pipeline-specific recommendations:")
        for rec in pipeline_recs:
            priority_icon = {'high': '🔴', 'medium': '🟡', 'low': '🟢', 'info': '🔵'}
            icon = priority_icon.get(rec.get('priority', 'info'), '🔵')
            print(f"  {icon} {rec['message']}")

print(f"\n📋 Next Steps:")
if assessment['quality_score'] >= 80:
    print(f"  1. Run full extraction on all 507 pages")
    print(f"  2. Integrate results into TypeScript database")
    print(f"  3. Implement recipe browsing interface")
else:
    print(f"  1. Address system issues identified above")
    print(f"  2. Test improvements on sample pages")
    print(f"  3. Re-run analysis to verify fixes")

print(f"\n🎉 Analysis Complete!")